In [6]:
import pandas as pd
import duckdb

print("Pandas version:", pd.__version__)
print("DuckDB version:", duckdb.__version__)

Pandas version: 2.3.3
DuckDB version: 1.4.2


# Dataset Occurrence u observaciones

In [7]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')

# Traer los primeros 5 registros a Pandas
df = con.execute("""
    SELECT *
    FROM read_csv_auto('~/Downloads/gbif_folder/Occurrence.txt')
""").fetchdf()  # o .df() también funciona
df.head(2)

# Ahora df es un DataFrame de Pandas

,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,verbatimEventDate,fieldNotes,behavior,sex,lifeStage,preparations,references,Associated Taxa,rightsHolder,license
0,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC9,HumanObservation,Wildlife sounds - Birds,None,None,Synallaxis,azarae,media,...,12-08-2002,two birds trip:http://www.cs.bris.ac.uk/home/p...,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,Bob Planqué,CC BY-NC
1,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC9,HumanObservation,Wildlife sounds - Birds,None,None,Synallaxis,azarae,media,...,12-08-2002,two birds trip:http://www.cs.bris.ac.uk/home/p...,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,Bob Planqué,CC BY-NC


In [9]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1638820 entries, 0 to 1638819
Data columns (total 37 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   id                     1638820 non-null  object        
 1   occurrenceID           1638820 non-null  object        
 2   catalogNumber          1638820 non-null  object        
 3   basisOfRecord          1638820 non-null  object        
 4   collectionCode         1638820 non-null  object        
 5   dynamicProperties      324456 non-null   object        
 6   otherCatalogNumbers    2174 non-null     object        
 7   genus                  1638820 non-null  object        
 8   specificEpithet        1638820 non-null  object        
 9   infraspecificEpithet   302249 non-null   object        
 10  scientificName         1638820 non-null  object        
 11  taxonRank              1638820 non-null  object        
 12  kingdom                16388

In [10]:
df.shape

(1638820, 37)

In [11]:
# Paises con mayor observacion de aves

import duckdb

con = duckdb.connect(database=':memory:')

# Contar ocurrencias por país y ordenar de mayor a menor
df_country_counts = con.execute("""
    SELECT country, COUNT(*) AS num_records
    FROM read_csv_auto('~/Downloads/gbif_folder/Occurrence.txt')
    GROUP BY country
    ORDER BY num_records DESC
""").fetchdf()  # o .df()

# Mostrar los primeros países con más registros
print(df_country_counts.head(20))

           country  num_records
0    United States       136821
1           Brazil       129488
2   United Kingdom       127166
3           France       113395
4         Colombia        68952
5          Ecuador        63606
6            Spain        62638
7          Germany        60480
8           Sweden        47175
9           Mexico        43797
10          Poland        39300
11     Netherlands        35448
12           China        34261
13           India        33301
14            Peru        31302
15       Australia        30171
16    South Africa        28304
17       Argentina        26280
18       Indonesia        22773
19        Malaysia        22524


In [12]:
#Eliminar Duplicados Biologicos

cols_bio = [
    'scientificName',
    'eventDate',
    'latitudeDecimal',
    'longitudeDecimal'
]

df_dups = df[df.duplicated(subset=cols_bio, keep=False)]
print("Duplicados biológicos encontrados:", len(df_dups))



Duplicados biológicos encontrados: 1612838


In [13]:
#Visualizacion de duplicado biologico 
df_dups.sort_values(cols_bio).head(5)


,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,verbatimEventDate,fieldNotes,behavior,sex,lifeStage,preparations,references,Associated Taxa,rightsHolder,license
43858,96758@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC96758,HumanObservation,Wildlife sounds - Birds,None,None,Abeillia,abeillei,None,...,2012-2-24,11:00 am,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,manuel Grosselet,CC BY-NC
43859,96758@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC96758,HumanObservation,Wildlife sounds - Birds,None,None,Abeillia,abeillei,None,...,2012-2-24,11:00 am,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,manuel Grosselet,CC BY-NC
1492110,168120@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC168120,HumanObservation,Wildlife sounds - Birds,None,None,Abeillia,abeillei,None,...,2014-02-16,animal seen:yes; playback used:no,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,Liliana Chavarria-Duriaux,CC BY-NC
1492111,168120@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC168120,HumanObservation,Wildlife sounds - Birds,None,None,Abeillia,abeillei,None,...,2014-02-16,animal seen:yes; playback used:no,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,Liliana Chavarria-Duriaux,CC BY-NC
1492114,168119@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC168119,HumanObservation,Wildlife sounds - Birds,None,None,Abeillia,abeillei,None,...,2014-02-16,Duet; animal seen:yes; playback used:no,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,has background sounds: Ramphastos sulfuratus|P...,Liliana Chavarria-Duriaux,CC BY-NC


In [14]:
# Cuantas veces se repite un evento biologico. Esto pasa por diversas fuentes en la misma base de datos
(
    df_dups
    .groupby(cols_bio)
    .size()
    .reset_index(name='n_repeticiones')
    .sort_values('n_repeticiones', ascending=False)
    .head(10)
)

,scientificName,eventDate,latitudeDecimal,longitudeDecimal,n_repeticiones
4798,Acrocephalus dumetorum,2021-06-24,52.477500,9.766100,154
173115,Egretta garzetta,2016-06-18,36.814900,-6.336600,146
31991,Anthus godlewskii,2022-09-09,47.746400,102.765300,142
174925,Elanus caeruleus,2017-01-13,37.382300,-6.136300,138
2028,Accipiter gentilis gentilis,2020-05-17,49.021100,9.024400,134
486277,Sternula albifrons,2016-06-18,36.814900,-6.336600,126
112131,Cistothorus apolinari apolinari,2000-06-01,4.603044,-74.204456,124
174924,Elanus caeruleus,2017-01-07,37.382300,-6.136300,122
272930,Locustella luscinioides,2020-05-30,48.702700,2.115600,106
415437,Pluvialis fulva,2022-09-05,48.913500,93.346400,98


In [15]:
#Eliminar duplicados biologicos

df_sin_dups_bio = df.drop_duplicates(subset=cols_bio)

print("Filas después de eliminar duplicados biológicos:",
      len(df_sin_dups_bio))

Filas después de eliminar duplicados biológicos: 620361


In [16]:
#Validacion

check = df_sin_dups_bio.duplicated(subset=cols_bio).sum()
print("Duplicados biológicos restantes:", check)

Duplicados biológicos restantes: 0


In [17]:
df_sin_dups_bio.shape

(620361, 37)

In [18]:
df_sin_dups_bio.head(1)

,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,verbatimEventDate,fieldNotes,behavior,sex,lifeStage,preparations,references,Associated Taxa,rightsHolder,license
0,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC9,HumanObservation,Wildlife sounds - Birds,None,None,Synallaxis,azarae,media,...,12-08-2002,two birds trip:http://www.cs.bris.ac.uk/home/p...,song,None,None,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,Bob Planqué,CC BY-NC


In [19]:
pd.set_option('display.max_rows', None)

#df_sin_dups_bio['scientificName'].value_counts()

In [20]:
import pandas as pd

# 1. Contar registros por scientificName
counts = df_sin_dups_bio['scientificName'].value_counts()

# 2. Filtrar solo los que tienen más de 800
names_gt_800 = counts[counts > 800].index.tolist()

# 3. Crear un dataframe NUEVO solo con esas especies
df_sin_dups_bio_800 = df_sin_dups_bio[df_sin_dups_bio['scientificName'].isin(names_gt_800)].copy()

print("Especies seleccionadas:", len(names_gt_800))
print("Filas en el nuevo dataframe:", len(df_sin_dups_bio_800))
df_sin_dups_bio_800.head(1)

Especies seleccionadas: 107
Filas en el nuevo dataframe: 163913


,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,verbatimEventDate,fieldNotes,behavior,sex,lifeStage,preparations,references,Associated Taxa,rightsHolder,license
45,999963@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC999963,HumanObservation,Wildlife sounds - Birds,"{""recordingDevice"":""Tascam x8"",""microphone"":""C...",None,Parus,major,None,...,2025-03-29,animal seen:yes; playback used:no,song,undetermined,adult,field recording,https://data.biodiversitydata.nl/xeno-canto/ob...,None,Bo Shunqi 薄顺奇,CC BY-NC


El dataset limpio biologicamente de todo el mundo, con 800 registros por especie, tiene 163913 filas y representa a 107 especies seleccionadas

# Datos de dataset Multimedia

In [21]:
import duckdb

con = duckdb.connect(database=':memory:')

# Traer los primeros 5 registros a Pandas
dfm = con.execute("""
    SELECT *
    FROM read_csv_auto('~/Downloads/gbif_folder/Multimedia.txt')
""").fetchdf()  # o .df() también funciona
dfm.head()

# Ahora df es un DataFrame de Pandas

,CoreId,associatedObservationReference,Identifier,type,Rating,rightsHolder,creator,accessURI,format,variantLiteral,description,caption,resourceCreationTechnique,captureDevice,physicalSetting,license
0,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,https://xeno-canto.org/sounds/uploaded/OH38YHK...,StillImage,<NA>,Stichting Xeno-canto voor Natuurgeluiden,Stichting Xeno-canto voor Natuurgeluiden,https://xeno-canto.org/sounds/uploaded/OH38YHK...,image/png,ac:MediumQuality,None,Oscillogram of the first ten seconds of the so...,None,None,None,CC BY-NC-SA 3.0
1,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,https://xeno-canto.org/sounds/uploaded/OH38YHK...,StillImage,<NA>,Stichting Xeno-canto voor Natuurgeluiden,Stichting Xeno-canto voor Natuurgeluiden,https://xeno-canto.org/sounds/uploaded/OH38YHK...,image/png,ac:MediumQuality,None,Oscillogram of the first ten seconds of the so...,None,None,None,CC BY-NC-SA 3.0
2,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,https://xeno-canto.org/sounds/uploaded/OH38YHK...,StillImage,<NA>,Stichting Xeno-canto voor Natuurgeluiden,Stichting Xeno-canto voor Natuurgeluiden,https://xeno-canto.org/sounds/uploaded/OH38YHK...,image/png,ac:MediumQuality,None,Spectrogram of the first ten seconds of the so...,None,None,None,CC BY-NC-SA 3.0
3,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,https://xeno-canto.org/sounds/uploaded/OH38YHK...,StillImage,<NA>,Stichting Xeno-canto voor Natuurgeluiden,Stichting Xeno-canto voor Natuurgeluiden,https://xeno-canto.org/sounds/uploaded/OH38YHK...,image/png,ac:MediumQuality,None,Spectrogram of the first ten seconds of the so...,None,None,None,CC BY-NC-SA 3.0
4,9@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,https://xeno-canto.org/sounds/uploaded/OH38YHK...,Sound,5,Bob Planqué,Bob Planqué,https://xeno-canto.org/sounds/uploaded/OH38YHK...,audio/mp3,ac:BestQuality,17 s,None,automatic recording: no; bitrate: 64000 bps; b...,None,Natural,CC BY-NC-SA 3.0


In [22]:
dfm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4895642 entries, 0 to 4895641
Data columns (total 16 columns):
 #   Column                          Dtype 
---  ------                          ----- 
 0   CoreId                          object
 1   associatedObservationReference  object
 2   Identifier                      object
 3   type                            object
 4   Rating                          Int64 
 5   rightsHolder                    object
 6   creator                         object
 7   accessURI                       object
 8   format                          object
 9   variantLiteral                  object
 10  description                     object
 11  caption                         object
 12  resourceCreationTechnique       object
 13  captureDevice                   object
 14  physicalSetting                 object
 15  license                         object
dtypes: Int64(1), object(15)
memory usage: 602.3+ MB


#Se une el dataset limpio biologicamente  con el archivo multimedia

In [23]:
import pandas as pd

# Cargar el dataset de Multimedia
df_multimedia = pd.read_csv('~/Downloads/gbif_folder/Multimedia.txt', sep=',')

# Hacer el merge con df_europe_america_800
df_merged = pd.merge(
    df_sin_dups_bio_800,
    df_multimedia,
    left_on='id',       # id de df_europe_america_800
    right_on='CoreId',  # CoreId de Multimedia
    how='left'          # left join para mantener todas las filas de df_europe_america_800plus
)

# Verificar
print("Filas del dataset df_sin_dups_bio_800", len(df_sin_dups_bio_800))
print("Filas del dataset df_merged después del merge:", len(df_merged))
df_merged.head(1)


Filas del dataset df_sin_dups_bio_800 163913
Filas del dataset df_merged después del merge: 945720


,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,creator,accessURI,format,variantLiteral,description,caption,resourceCreationTechnique,captureDevice,physicalSetting,license_y
0,999963@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC999963,HumanObservation,Wildlife sounds - Birds,"{""recordingDevice"":""Tascam x8"",""microphone"":""C...",None,Parus,major,None,...,Stichting Xeno-canto voor Natuurgeluiden,https://xeno-canto.org/sounds/uploaded/YNOAMCS...,image/png,ac:MediumQuality,NaN,Oscillogram of the first ten seconds of the so...,NaN,NaN,NaN,CC BY-NC-SA 4.0


In [24]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 945720 entries, 0 to 945719
Data columns (total 53 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              945720 non-null  object        
 1   occurrenceID                    945720 non-null  object        
 2   catalogNumber                   945720 non-null  object        
 3   basisOfRecord                   945720 non-null  object        
 4   collectionCode                  945720 non-null  object        
 5   dynamicProperties               232951 non-null  object        
 6   otherCatalogNumbers             1467 non-null    object        
 7   genus                           945720 non-null  object        
 8   specificEpithet                 945720 non-null  object        
 9   infraspecificEpithet            11553 non-null   object        
 10  scientificName                  945720 non-null  object 

In [25]:
# Filtrar filas donde format sea 'audio/mp3' o 'audio/wav'
df_filtrado = df_merged[df_merged['format'].isin(['audio/mp3', 'audio/wav'])].copy()

# Verificar
print("Filas con audio:", len(df_filtrado))
df_filtrado.head(1)

Filas con audio: 314593


,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,creator,accessURI,format,variantLiteral,description,caption,resourceCreationTechnique,captureDevice,physicalSetting,license_y
2,999963@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC999963,HumanObservation,Wildlife sounds - Birds,"{""recordingDevice"":""Tascam x8"",""microphone"":""C...",None,Parus,major,None,...,Bo Shunqi 薄顺奇,https://xeno-canto.org/sounds/uploaded/YNOAMCS...,audio/mp3,ac:BestQuality,44 s,NaN,automatic recording: no; bitrate: 192000 bps; ...,Tascam x8 | CICADA,Natural,CC BY-NC-SA 4.0


In [26]:
df_filtrado['scientificName'].value_counts()
df_filtrado['scientificName'].nunique()
# son 107 scientificname 

107

In [27]:
df_filtrado['country'].value_counts()

country
France                         52954
United Kingdom                 44279
Germany                        30122
Spain                          24932
Sweden                         21218
Netherlands                    20513
Poland                         20501
Belgium                        12943
Ireland                        10202
Italy                           9420
Portugal                        6174
Finland                         5479
Norway                          4806
United States                   4550
Estonia                         4397
Switzerland                     3522
Denmark                         3467
Austria                         2882
Ukraine                         2141
China                           1676
Croatia                         1546
India                           1269
Russian Federation              1173
Colombia                        1157
South Africa                    1122
Slovakia                        1082
Turkey                        

In [28]:
df_filtrado.shape

(314593, 53)

In [29]:
#Visualizacion de duplicados biologicos- nuevamente

cols_bio = [
    'scientificName',
    'eventDate',
    'latitudeDecimal',
    'longitudeDecimal'
]

#Marcar todos los duplicados 
df_dup_bio = df_filtrado[df_filtrado.duplicated(subset=cols_bio, keep=False)]

print("Duplicados biológicos encontrados:", len(df_dup_bio))

# Visualizacion
df_dup_bio_sorted = df_dup_bio.sort_values(cols_bio)
df_dup_bio_sorted.head(3)

Duplicados biológicos encontrados: 305122


,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,creator,accessURI,format,variantLiteral,description,caption,resourceCreationTechnique,captureDevice,physicalSetting,license_y
627017,519927@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC519927,HumanObservation,Wildlife sounds - Birds,None,None,Acrocephalus,arundinaceus,None,...,Leif Arvidsson,https://xeno-canto.org/sounds/uploaded/RVVFWWD...,audio/mp3,ac:BestQuality,239 s,NaN,automatic recording: no; bitrate: 320000 bps; ...,NaN,Natural,CC BY-NC-SA 4.0
627018,519927@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC519927,HumanObservation,Wildlife sounds - Birds,None,None,Acrocephalus,arundinaceus,None,...,Leif Arvidsson,https://xeno-canto.org/sounds/uploaded/RVVFWWD...,audio/mp3,ac:BestQuality,239 s,NaN,automatic recording: no; bitrate: 320000 bps; ...,NaN,Natural,CC BY-NC-SA 4.0
627035,519907@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC519907,HumanObservation,Wildlife sounds - Birds,None,None,Acrocephalus,arundinaceus,None,...,Leif Arvidsson,https://xeno-canto.org/sounds/uploaded/RVVFWWD...,audio/mp3,ac:BestQuality,55 s,NaN,automatic recording: no; bitrate: 320000 bps; ...,NaN,Natural,CC BY-NC-SA 4.0


In [31]:
df_limpio = df_filtrado.drop_duplicates(subset=cols_bio)
print("Filas sin duplicados biológicos:", len(df_limpio))
print(df_limpio.shape)

Filas sin duplicados biológicos: 162032
(162032, 53)


In [32]:
df_limpio.shape

(162032, 53)

In [33]:
df_limpio['country'].value_counts()

country
France                         27388
United Kingdom                 22899
Germany                        15247
Spain                          12770
Sweden                         11023
Netherlands                    10408
Poland                         10353
Belgium                         6721
Ireland                         5457
Italy                           4899
Portugal                        3113
Finland                         2846
Norway                          2499
United States                   2309
Estonia                         2300
Switzerland                     1908
Denmark                         1834
Austria                         1478
Ukraine                         1077
China                            885
Croatia                          781
India                            649
Russian Federation               598
Colombia                         583
South Africa                     565
Slovakia                         551
Turkey                        

In [59]:
df_limpio['scientificName'].value_counts()

scientificName
Mystery mystery              15059
Parus major                   4269
Turdus merula                 4216
Erithacus rubecula            3978
Fringilla coelebs             3595
                             ...  
Picus viridis                  817
Acrocephalus arundinaceus      813
Anser anser                    807
Tringa glareola                795
Streptopelia decaocto          794
Name: count, Length: 107, dtype: int64

In [34]:
#Luego del Merge, se vuelve a filtrar las especies con mas de 800 registros

import pandas as pd

# 1. Contar registros por scientificName
counts = df_limpio['scientificName'].value_counts()

# 2. Filtrar solo los que tienen más de 800
names = counts[counts > 800].index.tolist()

# 3. Crear un dataframe NUEVO solo con esas especies
df_final = df_limpio[df_limpio['scientificName'].isin(names)].copy()

print("Especies seleccionadas:", len(names))
print("Filas en el nuevo dataframe:", len(df_final))
df_final.head(1)

Especies seleccionadas: 105
Filas en el nuevo dataframe: 160443


,id,occurrenceID,catalogNumber,basisOfRecord,collectionCode,dynamicProperties,otherCatalogNumbers,genus,specificEpithet,infraspecificEpithet,...,creator,accessURI,format,variantLiteral,description,caption,resourceCreationTechnique,captureDevice,physicalSetting,license_y
2,999963@XC,https://data.biodiversitydata.nl/xeno-canto/ob...,XC999963,HumanObservation,Wildlife sounds - Birds,"{""recordingDevice"":""Tascam x8"",""microphone"":""C...",None,Parus,major,None,...,Bo Shunqi 薄顺奇,https://xeno-canto.org/sounds/uploaded/YNOAMCS...,audio/mp3,ac:BestQuality,44 s,NaN,automatic recording: no; bitrate: 192000 bps; ...,Tascam x8 | CICADA,Natural,CC BY-NC-SA 4.0


In [35]:
df_final['scientificName'].value_counts() #Lista de especies con mas de 800 archivos de sonidos para entrenar

scientificName
Mystery mystery                  15059
Parus major                       4269
Turdus merula                     4216
Erithacus rubecula                3978
Fringilla coelebs                 3595
Sylvia atricapilla                3303
Turdus philomelos                 3254
Phylloscopus collybita            3200
Troglodytes troglodytes           2668
Strix aluco                       2391
Rallus aquaticus                  2320
Cyanistes caeruleus               2249
Turdus iliacus                    2104
Gallinula chloropus               2027
Emberiza citrinella               1966
Dendrocopos major                 1865
Fulica atra                       1858
Loxia curvirostra                 1762
Luscinia megarhynchos             1731
Alauda arvensis                   1726
Anthus trivialis                  1712
Phylloscopus trochilus            1682
Emberiza calandra                 1675
Curruca communis                  1647
Actitis hypoleucos                1645
Prunella m

In [62]:
df_final['country'].value_counts()

country
France            27149
United Kingdom    22760
Germany           15156
Spain             12633
Sweden            10945
                  ...  
Iraq                  1
Nauru                 1
Qatar                 1
Antarctica            1
Sao Tome              1
Name: count, Length: 167, dtype: int64

In [36]:
df_final.shape

(160443, 53)

In [37]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 160443 entries, 2 to 945719
Data columns (total 53 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              160443 non-null  object        
 1   occurrenceID                    160443 non-null  object        
 2   catalogNumber                   160443 non-null  object        
 3   basisOfRecord                   160443 non-null  object        
 4   collectionCode                  160443 non-null  object        
 5   dynamicProperties               42018 non-null   object        
 6   otherCatalogNumbers             260 non-null     object        
 7   genus                           160443 non-null  object        
 8   specificEpithet                 160443 non-null  object        
 9   infraspecificEpithet            1965 non-null    object        
 10  scientificName                  160443 non-null  object      

In [39]:
df_final.to_csv('~/Desktop/df_final.csv', index=False)

Finalmente, el Dataset df_final tiene:

1. Aves de todo el mundo 
2. Nombres de Especies con mas de 800 registros 
3. Solamente filas con archivos de sonidos mp3 o wav
4. Tiene filas 160 444 y 53 columnas luego de una limpieza biologica